In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/cleaned/online_retail_cleaned.csv",
    parse_dates=["InvoiceDate", "Month_Start"]
)

sales_df = df[
    (df["Is_Return"] == False) &
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (df["Customer_ID"].notna())
].copy()

print("Sales records:", len(sales_df))
print("Customers:", sales_df["Customer_ID"].nunique())

Sales records: 779425
Customers: 5878


In [2]:
customer_orders = (
    sales_df
    .groupby("Customer_ID")["Invoice"]
    .nunique()
)

customer_type = pd.cut(
    customer_orders,
    bins=[0, 1, np.inf],
    labels=["One-Time Customer", "Repeat Customer"]
)

customer_type_counts = customer_type.value_counts()

customer_type_counts

Invoice
Repeat Customer      4255
One-Time Customer    1623
Name: count, dtype: int64

In [3]:
customer_type_percentage = (
    customer_type_counts /
    customer_type_counts.sum() * 100
).round(2)

customer_type_percentage

Invoice
Repeat Customer      72.39
One-Time Customer    27.61
Name: count, dtype: float64

In [4]:
customer_clv = (
    sales_df
    .groupby("Customer_ID")
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Orders=("Invoice", "nunique"),
        Total_Quantity=("Quantity", "sum")
    )
    .reset_index()
)

In [5]:
customer_clv["Average_Order_Value"] = (
    customer_clv["Total_Revenue"] /
    customer_clv["Total_Orders"]
)

In [6]:
customer_clv["Purchase_Frequency"] = (
    customer_clv["Total_Orders"]
)

In [7]:
customer_clv["Historical_CLV"] = (
    customer_clv["Total_Revenue"]
)

In [8]:
customer_clv.sort_values(
    "Historical_CLV",
    ascending=False
).head(20)

,Customer_ID,Total_Revenue,Total_Orders,Total_Quantity,Average_Order_Value,Purchase_Frequency,Historical_CLV
5692,18102.0,580987.04,145,181645,4006.807172,145,580987.04
2277,14646.0,528602.52,151,367193,3500.678940,151,528602.52
1789,14156.0,313437.62,156,164325,2009.215513,156,313437.62
2538,14911.0,291420.81,398,147972,732.213090,398,291420.81
5050,17450.0,244784.25,51,83914,4799.691176,51,244784.25
1331,13694.0,195640.69,143,188201,1368.116713,143,195640.69
5109,17511.0,172132.87,60,117174,2868.881167,60,172132.87
4061,16446.0,168472.50,2,80997,84236.250000,2,168472.50
4295,16684.0,147142.77,55,104810,2675.323091,55,147142.77
68,12415.0,144458.37,28,91447,5159.227500,28,144458.37


In [9]:
pareto = customer_clv.sort_values(
    "Total_Revenue",
    ascending=False
).copy()

In [10]:
pareto["Cumulative_Revenue"] = (
    pareto["Total_Revenue"].cumsum()
)

In [11]:
total_customer_revenue = pareto["Total_Revenue"].sum()

In [12]:
pareto["Cumulative_Revenue_Percentage"] = (
    pareto["Cumulative_Revenue"] /
    total_customer_revenue * 100
)

In [13]:
customers_for_80 = (
    pareto["Cumulative_Revenue_Percentage"] <= 80
).sum()

total_customers = len(pareto)

percentage_customers = (
    customers_for_80 /
    total_customers * 100
)

print("Customers generating first 80% of revenue:", customers_for_80)
print("Percentage of customers:", round(percentage_customers, 2), "%")

Customers generating first 80% of revenue: 1353
Percentage of customers: 23.02 %


In [14]:
first_purchase = (
    sales_df
    .groupby("Customer_ID")["InvoiceDate"]
    .min()
    .reset_index()
)

first_purchase.columns = [
    "Customer_ID",
    "First_Purchase_Date"
]

In [15]:
first_purchase["Cohort_Month"] = (
    first_purchase["First_Purchase_Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [16]:
sales_cohort = sales_df.merge(
    first_purchase[
        ["Customer_ID", "Cohort_Month"]
    ],
    on="Customer_ID",
    how="left"
)

In [17]:
sales_cohort["Purchase_Month"] = (
    sales_cohort["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [18]:
sales_cohort["Cohort_Index"] = (
    (sales_cohort["Purchase_Month"].dt.year -
     sales_cohort["Cohort_Month"].dt.year) * 12
    +
    (sales_cohort["Purchase_Month"].dt.month -
     sales_cohort["Cohort_Month"].dt.month)
    + 1
)

In [19]:
sales_cohort[
    [
        "Customer_ID",
        "Cohort_Month",
        "Purchase_Month",
        "Cohort_Index"
    ]
].head()

,Customer_ID,Cohort_Month,Purchase_Month,Cohort_Index
0,13085.0,2009-12-01,2009-12-01,1
1,13085.0,2009-12-01,2009-12-01,1
2,13085.0,2009-12-01,2009-12-01,1
3,13085.0,2009-12-01,2009-12-01,1
4,13085.0,2009-12-01,2009-12-01,1


In [20]:
cohort_data = (
    sales_cohort
    .groupby(
        ["Cohort_Month", "Cohort_Index"]
    )["Customer_ID"]
    .nunique()
    .reset_index()
)

In [21]:
cohort_table = cohort_data.pivot(
    index="Cohort_Month",
    columns="Cohort_Index",
    values="Customer_ID"
)

In [22]:
cohort_size = cohort_table.iloc[:, 0]

cohort_retention = (
    cohort_table
    .div(cohort_size, axis=0)
    * 100
)

cohort_retention.round(2)

Cohort_Index,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,25
Cohort_Month,,,,,,,,,,,,,,,,,,,,,
2009-12-01,100.0,35.29,33.40,42.51,38.01,35.92,37.70,34.24,33.61,36.23,...,30.26,26.28,30.26,28.27,25.97,25.55,31.52,30.47,40.73,19.69
2010-01-01,100.0,20.63,31.07,30.55,26.37,30.03,25.85,22.98,27.94,31.85,...,15.14,23.50,19.84,18.54,19.58,24.28,19.32,24.54,5.74,NaN
2010-02-01,100.0,23.80,22.46,29.14,24.60,20.05,19.25,28.61,25.40,27.54,...,20.05,16.04,16.31,14.44,22.99,22.99,16.31,5.88,NaN,NaN
2010-03-01,100.0,18.96,23.02,24.15,23.25,20.32,24.60,30.25,27.54,10.84,...,16.93,17.38,15.58,17.61,20.09,21.22,7.90,NaN,NaN,NaN
2010-04-01,100.0,19.39,19.39,16.33,18.37,22.45,27.55,26.19,10.54,10.88,...,15.65,13.95,14.97,18.03,22.45,5.78,NaN,NaN,NaN,NaN
2010-05-01,100.0,15.75,16.93,17.32,17.72,25.59,21.26,12.60,5.91,8.27,...,12.60,13.78,16.54,15.35,4.72,NaN,NaN,NaN,NaN,NaN
2010-06-01,100.0,17.41,18.89,20.37,22.96,28.52,12.59,8.89,8.15,11.85,...,12.22,13.33,20.37,5.19,NaN,NaN,NaN,NaN,NaN,NaN
2010-07-01,100.0,15.59,18.28,29.57,29.03,13.98,11.29,14.52,14.52,11.29,...,17.20,23.66,8.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08-01,100.0,20.37,29.63,32.10,17.28,11.73,9.88,12.35,13.58,12.96,...,19.75,6.79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
returns_df = df[
    df["Is_Return"] == True
].copy()

In [24]:
product_returns = (
    returns_df
    .groupby("Description")["Quantity"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

In [25]:
product_returns.head(20)

Description
PAPER CRAFT , LITTLE BIRDIE            80995
MEDIUM CERAMIC TOP STORAGE JAR         74494
?                                      36888
given away                             20000
printing smudges/thrown away           19200
missing                                16467
Unsaleable, destroyed.                 15644
ebay sales                             13630
check                                  13274
Given away                             10200
ROTATING SILVER ANGELS T-LIGHT HLDR     9381
Printing smudges/thrown away            9058
Zebra invcing error                     9000
damages                                 8083
Damaged                                 8013
SET/6 FRUIT SALAD PAPER CUPS            7140
Ebay sales by the box.                  7100
SET/6 FRUIT SALAD  PAPER PLATES         7008
Manual                                  5443
throw away                              5368
Name: Quantity, dtype: int64

In [26]:
product_sales = (
    sales_df
    .groupby("Description")["Quantity"]
    .sum()
)

product_return_quantity = (
    returns_df
    .groupby("Description")["Quantity"]
    .sum()
    .abs()
)

In [27]:
product_return_analysis = pd.DataFrame({
    "Units_Sold": product_sales,
    "Units_Returned": product_return_quantity
}).fillna(0)

In [28]:
product_return_analysis["Return_Rate"] = (
    product_return_analysis["Units_Returned"] /
    product_return_analysis["Units_Sold"] * 100
)

In [29]:
product_return_analysis = (
    product_return_analysis[
        product_return_analysis["Units_Sold"] >= 100
    ]
    .sort_values(
        "Return_Rate",
        ascending=False
    )
)

In [30]:
product_return_analysis.head(20)

,Units_Sold,Units_Returned,Return_Rate
Description,,,
Discount,196.0,3064.0,1563.265306
WHITE CHERRY LIGHTS,962.0,1072.0,111.434511
BLACK CHERRY LIGHTS,231.0,248.0,107.359307
"PAPER CRAFT , LITTLE BIRDIE",80995.0,80995.0,100.000000
LITTLE FLOWER SHOPPER BAG,400.0,400.0,100.000000
RED POLKADOT PUDDING BOWL,3708.0,3648.0,98.381877
MULTICOLOUR POLKADOT PLATE,696.0,684.0,98.275862
SET OF 6 DOTS CHOPSTICKS,185.0,181.0,97.837838
BLUE POLKADOT PUDDING BOWL,2564.0,2496.0,97.347894


In [31]:
customer_clv.to_csv(
    "../data/cleaned/customer_clv.csv",
    index=False
)

pareto.to_csv(
    "../data/cleaned/customer_pareto.csv",
    index=False
)

cohort_retention.to_csv(
    "../data/cleaned/cohort_retention.csv"
)

product_return_analysis.to_csv(
    "../data/cleaned/product_return_analysis.csv"
)

print("Advanced analytics saved successfully!")

Advanced analytics saved successfully!


In [1]:
import os

files = os.listdir("../data/cleaned")

print("\n".join(files))

.ipynb_checkpoints
cohort_retention.csv
country_revenue.csv
customer_clv.csv
customer_pareto.csv
customer_revenue.csv
customer_rfm.csv
customer_segments.csv
monthly_revenue.csv
online_retail_cleaned.csv
product_return_analysis.csv
sql_country_sales.csv
sql_monthly_growth.csv
sql_rfm_segments.csv
sql_top_customers.csv
sql_top_products.csv
top_products_revenue.csv
